In [ ]:
!pip install -q ultralytics roboflow
print("Download Complete ✅")

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("cctv-v79fb").project("pothole-4ljdw")
version = project.version(1)
dataset = version.download("yolo26")
print("Dataset Downloaded ✅")

In [ ]:
from google.colab import drive
import random, shutil, zipfile
from pathlib import Path

# Mount google drive
drive.mount('/content/drive')

# Unzip directly from my drive
zip_path = "/content/drive/MyDrive/roboflow_batch1.zip"
bg_dir = Path("/content/bg_batch")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(bg_dir)

# Grab all extracted .jpg files (ignores macOS __MACOSX metadata folder)
imgs = [p for p in bg_dir.rglob("*.jpg") if "__MACOSX" not in p.parts]
random.seed(42)
random.shuffle(imgs)

# Split: 50 to valid, the rest (~300) to train
val_imgs, train_imgs = imgs[:50], imgs[50:]

def add_backgrounds(file_list, split):
    img_dst = Path(f"{dataset.location}/{split}/images")
    lbl_dst = Path(f"{dataset.location}/{split}/labels")
    for p in file_list:
        shutil.copy2(p, img_dst / p.name)
        (lbl_dst / f"{p.stem}.txt").write_text("")  # Empty .txt = Null/Background

add_backgrounds(train_imgs, "train")
add_backgrounds(val_imgs, "valid")

print(f"Added {len(train_imgs)} train + {len(val_imgs)} valid background frames ✅")

In [ ]:
!ls -lh /content/roboflow_batch1.zip

In [ ]:
import yaml
from ultralytics import YOLO

#Rename the class from the timestamp string to 'pothole'
yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, 'r') as f:
  data = yaml.safe_load(f)
data["names"] = ["pothole"]
with open(yaml_path, 'w') as f:
  yaml.dump(data, f)

#Train YOLO26 nano baseline
model = YOLO("yolo26n.pt")
model.train(data=yaml_path, epochs=50, imgsz=640)

In [ ]:
from google.colab import files
files.download("/content/runs/detect/train/weights/best.pt")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob

# 1. Load your trained v2 weights
model = YOLO("/content/runs/detect/train/weights/best.pt")

# 2. Run prediction on your extracted frames folder
results = model.predict(
    source="/content/drive/MyDrive/RoadWatchData/extracted_frames_test", # Update to your folder path in Colab
    conf=0.25,
    save=True,
    imgsz=640
)

print(f"Processed {len(results)} images. Displaying predictions:")

# 3. Display the annotated images with bounding boxes right inside Colab
for img_path in glob.glob(f"{results[0].save_dir}/*.jpg")[:5]:
    display(Image(filename=img_path, width=600))

In [ ]:
from pathlib import Path
from IPython.display import Image, display

# Filter to only results that have at least 1 bounding box
detected = [r for r in results if len(r.boxes) > 0]
print(f"Found detections in {len(detected)} out of {len(results)} images.")

for r in detected[:10]:  # Show up to 10 images with boxes
    img_name = Path(r.path).name
    print(f"{img_name} -> {len(r.boxes)} box(es), confs: {[round(float(c), 2) for c in r.boxes.conf]}")
    display(Image(filename=f"{r.save_dir}/{img_name}", width=600))